In [1]:
import pandas as pd
import glob
import os
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from translatepy import Translator
from collections import Counter

import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib import cm

In [ ]:
folder_path = "../Step_1/Extracted_Jobs"

In [4]:
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

In [ ]:
dataframes = []

for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file,  sep=";", on_bad_lines='skip', engine='python')
        dataframes.append(df)
    except Exception as e:
        print(f"Error while reading {csv_file} :", e)

if dataframes:
    df_total = pd.concat(dataframes, ignore_index=True)
    print("Merge successful!")
else:
    print("No file read successfully, cannot merge.")

df_total_clean = df_total.dropna(how='all')  

In [ ]:
# Translation function: all job titles are translated to English so that some jobs don't appear in French and others in English, which would complicate the next steps.

def translate_to_english(text):
    translator = Translator()
    try:
        translated = translator.translate(text, "English")
        return translated.result
    except Exception as e:
        print(f"Error during translation: {e}")
        return text 


def translate_dataframe(df, columns_to_translate):
    for column in columns_to_translate:
        if column in df.columns:
            df[column] = df[column].apply(lambda x: translate_to_english(str(x)) if pd.notna(x) else x)
    return df

columns_to_translate = ['job_title']  

df_total_clean = translate_dataframe(df_total_clean, columns_to_translate)

In [ ]:
df_total_clean

In [8]:
# Drop unneeded columns

df_total_clean = df_total_clean.drop(columns=[ 'link','time_posted'])

In [ ]:
#null values

df_total_clean.isna().sum()

In [10]:
#drop the missing values

df_total_clean=df_total_clean.dropna()

In [ ]:
# Duplicate data (x duplicates over y columns)

df_total_clean[df_total_clean.duplicated(subset=["job_title","company_name", "location"])].shape

In [ ]:
# Words to exclude and seniority keywords
exclude_keywords = {'engineer', 'developer', 'software'}
expertise_keywords = {'senior', 'lead', 'junior', 'intern', 'manager'}

# Clean and extract words
job_keywords =df_total_clean['job_title'].str.lower().str.replace(r'[^a-zA-Z\s]', '', regex=True)
words = job_keywords.str.split().sum()
word_freq = Counter(words)

# Split the words
expertise_counts = {k: word_freq[k] for k in expertise_keywords if k in word_freq}
remaining_words = {k: v for k, v in word_freq.items() if k not in exclude_keywords and k not in expertise_keywords}

# Pie chart data for seniority keywords
df_expertise = pd.DataFrame(list(expertise_counts.items()), columns=["expertise", "count"])
df_expertise = df_expertise.sort_values(by="count", ascending=False)

df_expertise

In [13]:
# Clean job titles

def clean_job_title(title):

    title = str(title).strip().lower()
    title = title.replace('’', "'").replace('–', '-')
    title = ''.join(e for e in title if e.isalnum() or e.isspace())
    return title

In [ ]:
# Apply the cleaning to the DataFrame

if 'job_title' in df_total_clean.columns :
    df_total_clean['job_title'] = df_total_clean['job_title'].apply(clean_job_title)

    print("Job titles cleaned successfully")
else:
    print("column must exist")

In [ ]:
job_titles = df_total_clean['job_title']

useful_words = {
    'python', 'java', 'cloudops', 'devsecops', 'kubernetes','fullstack', 'qa', 'hf',
    'frontend','front-end', 'front end', 'backend', 'back-end', 'back end', 'ai', 'ml', 
    'data', 'devops', 'flutter', 'android', 'ios','mlops', 'founding', 'nlp', 'microservices', 
    'security', 'database', 'machine learning'
}

# Initialize the vectorizer
vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 1)) 

# Turn the titles into a count matrix
X = vectorizer.fit_transform(job_titles)

# Get the words
terms = vectorizer.get_feature_names_out()

# Compute the frequency of each term
frequencies = np.array(X.sum(axis=0)).flatten()

# Create the DataFrame
df_keywords = pd.DataFrame({'keyword': terms, 'frequency': frequencies})

# Group similar keywords and sum their frequencies
df_keywords = df_keywords.groupby('keyword')['frequency'].sum().reset_index()

# Words to ignore (not useful)
ignored_words = [
    'engineer', 'engineers', 'software engineer', 'developer', 'stack', 
    'senior', 'junior', 'lead', 'senior software engineer', 'intern', 'manager',
    'remote', 'platform', 'software', 'software developer', 'senior software', 'development'
]

# Filter out the ignored words
df_keywords = df_keywords[~df_keywords['keyword'].isin(ignored_words)]

# Keep only the useful words
df_useful_keywords = df_keywords[df_keywords['keyword'].isin(useful_words)]

# Sort by descending frequency
df_useful_keywords = df_useful_keywords.sort_values(by='frequency', ascending=False).reset_index(drop=True)

# Show the top 30 useful keywords
df_useful_keywords.head(30)

In [ ]:
# Global style
sns.set(style="whitegrid")
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': '#f4f4f4',  # soft gray background
    'axes.facecolor': 'white'
})

# Figure
fig = plt.figure(figsize=(16, 9))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1], wspace=0.3, hspace=0.3)

# Function to draw a card frame around each chart
def draw_card(ax, fig, pad=0.015, radius=0.02):
    bbox = ax.get_position()
    rect = patches.FancyBboxPatch(
        (bbox.x0 - pad, bbox.y0 - pad),
        bbox.width + 2 * pad,
        bbox.height + 2 * pad,
        boxstyle=f"round,pad=0.02,rounding_size={radius}",
        linewidth=1.2,
        edgecolor='#cccccc',
        facecolor='white',
        zorder=0
    )
    fig.patches.append(rect)

# Bar chart
ax_bar = fig.add_subplot(gs[:, 0])
top_keywords = df_useful_keywords.sort_values(by='frequency', ascending=False).head(10)
sns.barplot(
    y='keyword',
    x='frequency',
    data=top_keywords,
    ax=ax_bar,
    palette="Blues_r"
)
ax_bar.set_title("Top 10 Job Keywords")
ax_bar.set_xlabel("Frequency")
ax_bar.set_ylabel("Keyword")
ax_bar.spines[['top', 'right']].set_visible(False)
ax_bar.grid(axis='x', linestyle='--', linewidth=0.5, color='#dddddd')
draw_card(ax_bar, fig)

# Pie chart: companies
ax_pie1 = fig.add_subplot(gs[0, 1])
explode = [0.05] + [0.03]*9
df_total_clean["company_name"].value_counts().head(10).plot.pie(
    ax=ax_pie1,
    explode=explode,
    colors=sns.color_palette("pastel"),
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 10, 'color': '#333'}
)
ax_pie1.set_title("Top 10 Companies", pad=15)
ax_pie1.set_ylabel('')
draw_card(ax_pie1, fig)

# Pie chart: seniority
ax_pie2 = fig.add_subplot(gs[1, 1])
ax_pie2.pie(
    df_expertise["count"],
    labels=df_expertise["expertise"],
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette("Set3"),
    textprops={'fontsize': 10, 'color': '#333'}
)
ax_pie2.set_title("Expertise Levels in Job Titles", pad=15)
ax_pie2.axis('equal')
draw_card(ax_pie2, fig)


fig.suptitle("Dashboard: Job Market Analysis", fontsize=18, weight='bold', color="#333", y=0.98)

plt.show()